# Ejemplo: Red Neuronal con TensorFlow

Este notebook demuestra cómo entrenar una red neuronal simple para clasificación usando TensorFlow (standalone).

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. Preparar datos con tf.data

In [ ]:
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_classes=3, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Crear Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(32)

print(f"Train batches: {len(train_dataset)}")

## 2. Construir modelo (Functional API)

In [ ]:
inputs = tf.keras.Input(shape=(20,), name='input')
x = tf.keras.layers.Dense(64, activation='relu', name='dense1')(inputs)
x = tf.keras.layers.Dropout(0.3, name='dropout1')(x)
x = tf.keras.layers.Dense(32, activation='relu', name='dense2')(x)
outputs = tf.keras.layers.Dense(3, activation='softmax', name='output')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='simple_nn')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. Custom Training Loop

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-3)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        loss = loss_fn(labels, predictions)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

epochs = 20
for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in train_dataset:
        loss = train_step(batch_x, batch_y)
        total_loss += loss
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_dataset):.4f}")

## 4. Evaluar

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.4f}")

## 5. Guardar modelo (SavedModel)

In [ ]:
# Guardar en formato SavedModel
model.save('my_model')
print("Modelo guardado en formato SavedModel")

# También en formato Keras
model.save('model_tensorflow.keras')
print("Modelo guardado en model_tensorflow.keras")

## 6. TensorFlow Lite

In [ ]:
# Convertir a TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Modelo TFLite guardado en model.tflite")